## Silver · Data profile of the raw trips
Profile `01_bronze.yellow_trips_raw` **before** writing any cleaning rule.
Every DQ rule in `02_silver/fact_trips` is backed by one of these checks and its row count.

Run the cells top to bottom: each check shows its result and adds it to the report;
the last cell saves the report to `outputs/profile_report.md`.

In [0]:
%run ../../config/config

In [0]:
%run ../../utils/zsi_functions

### Setup

In [0]:
import os

from pyspark.sql import functions as F

# Raw trips + helper columns from add_dq_columns():
#   _pu_nonexistent / _do_nonexistent : timestamp falls in the DST gap (local time that never existed)
#   _true_min                         : real elapsed minutes (DST-aware)
#   _naive_min                        : dropoff - pickup on the wall clock
#   _dup_rn                           : 1 for the first copy of a row, 2+ for exact duplicates
raw = spark.table(tbl("01_bronze", "yellow_trips_raw"))
trips = add_dq_columns(raw).cache()

source_columns = [c for c in raw.columns if c != "processing_timestamp"]
lookup_ids = [row.location_id for row in spark.table(tbl("01_bronze", "taxi_zone_lookup_raw")).select(F.col("LocationID").cast("int").alias("location_id")).collect()]

print(f"raw rows: {trips.count():,}")

In [0]:
# Column shortcuts used in the checks
pickup_ts       = F.col("tpep_pickup_datetime")
pu_nonexistent  = F.col("_pu_nonexistent")
do_nonexistent  = F.col("_do_nonexistent")
true_min        = F.col("_true_min")
naive_min       = F.col("_naive_min")
pu_location     = F.col("PULocationID")
do_location     = F.col("DOLocationID")
fare_amount     = F.col("fare_amount")
total_amount    = F.col("total_amount")
trip_distance   = F.col("trip_distance")


def count_if(condition, name):
    """Number of rows where the condition is true (NULL counts as false)."""
    return F.sum(F.when(condition, 1).otherwise(0)).alias(name)


report = {}   # check title -> markdown; re-running a cell replaces its entry


def add_to_report(title, df):
    """Store one check's result for the report saved in the last cell."""
    report[title] = f"## {title}\n\n{md_table(to_pandas_safe(df))}"

### A. Volume and duplicates

#### 1. Raw row count

In [0]:
raw_row_count = trips.agg(F.count("*").alias("raw_rows"))

display(raw_row_count)
add_to_report("1. Raw row count", raw_row_count)

#### 2. NULLs per column

In [0]:
nulls_per_column = trips.agg(*[count_if(F.col(c).isNull(), c) for c in source_columns])

display(nulls_per_column)
add_to_report("2. NULLs per column", nulls_per_column)

#### 3. Exact duplicate rows
Rows identical in every source column (NULLs compare equal). `extra_copies` = rows a dedup would remove.

In [0]:
exact_duplicates = trips.agg(count_if(F.col("_dup_rn") > 1, "extra_copies"))

display(exact_duplicates)
add_to_report("3. Exact duplicate rows", exact_duplicates)

### B. Timestamps

#### 4. Pickup month distribution
A March file should contain March pickups; other months point to wrong vendor clocks.

In [0]:
pickup_months = (
    trips
    .groupBy(F.date_format(pickup_ts, "yyyy-MM").alias("pickup_month"))
    .agg(F.count("*").alias("trips"))
    .orderBy("pickup_month")
)

display(pickup_months)
add_to_report("4. Pickup month distribution", pickup_months)

#### 5. Pickups outside the processing period

In [0]:
outside_period = trips.agg(
    count_if(pickup_ts < F.to_timestamp(F.lit(PERIOD_START)), "before_period"),
    count_if(pickup_ts >= F.to_timestamp(F.lit(PERIOD_END)), "after_period"),
)

display(outside_period)
add_to_report("5. Pickups outside the period", outside_period)

#### 6. Timestamps at a nonexistent local time
DST started 2025-03-09 02:00 → clocks jumped to 03:00, so 02:00–02:59 never existed in New York.

In [0]:
dst_gap = trips.agg(
    count_if(pu_nonexistent, "pickup"),
    count_if(do_nonexistent, "dropoff"),
    count_if(pu_nonexistent | do_nonexistent, "either_rows"),
)

display(dst_gap)
add_to_report("6. Timestamps at a nonexistent local time", dst_gap)

#### 7. DST-gap rows by vendor

In [0]:
dst_gap_by_vendor = (
    trips
    .filter(pu_nonexistent | do_nonexistent)
    .groupBy("VendorID")
    .agg(F.count("*").alias("trips"))
    .orderBy("VendorID")
)

display(dst_gap_by_vendor)
add_to_report("7. DST-gap rows by vendor", dst_gap_by_vendor)

### C. Durations

#### 8. True vs naive duration
Trips whose real elapsed time differs from wall-clock `dropoff - pickup` (trips spanning the DST jump).

In [0]:
duration_difference = (
    trips
    .filter(~pu_nonexistent & ~do_nonexistent)
    .filter(F.abs(naive_min - true_min) > 1e-9)
    .groupBy(F.round(naive_min - true_min, 4).alias("naive_minus_true_min"))
    .agg(F.count("*").alias("trips"))
    .orderBy("naive_minus_true_min")
)

display(duration_difference)
add_to_report("8. True vs naive duration", duration_difference)

#### 9. True duration buckets

In [0]:
duration_bucket = (
    F.when(true_min.isNull(), "0. null")
     .when(true_min < 0,      "1. negative")
     .when(true_min == 0,     "2. zero")
     .when(true_min < 1,      "3. under 1 min")
     .when(true_min <= 180,   "4. 1 min - 3 h")
     .when(true_min <= 360,   "5. 3 h - 6 h")
     .when(true_min <= 1440,  "6. 6 h - 24 h")
     .otherwise(              "7. over 24 h")
)

duration_buckets = (
    trips
    .groupBy(duration_bucket.alias("bucket"))
    .agg(F.count("*").alias("trips"))
    .orderBy("bucket")
)

display(duration_buckets)
add_to_report("9. True duration buckets", duration_buckets)

#### 10. Long durations by hour
Shape of the tail above `MAX_DURATION_MIN`; a cluster near 24 h means meters left running.

In [0]:
long_durations = (
    trips
    .filter(true_min > MAX_DURATION_MIN)
    .groupBy(F.floor(true_min / 60).cast("int").alias("hours"))
    .agg(F.count("*").alias("trips"))
    .orderBy("hours")
)

display(long_durations)
add_to_report("10. Long durations by hour", long_durations)

### D. Locations

#### 11. Location IDs
NULL or not in the lookup = invalid. 264 / 265 are valid lookup rows but not physical zones.

In [0]:
location_checks = trips.agg(
    count_if(pu_location.isNull(),            "pu_null"),
    count_if(~pu_location.isin(lookup_ids),   "pu_not_in_lookup"),
    count_if(do_location.isNull(),            "do_null"),
    count_if(~do_location.isin(lookup_ids),   "do_not_in_lookup"),
    count_if(pu_location.isin(264, 265),      "pu_unknown_zone"),
    count_if(do_location.isin(264, 265),      "do_unknown_zone"),
)

display(location_checks)
add_to_report("11. Location IDs", location_checks)

### E. Amounts

#### 12. Negative amounts

In [0]:
negative_amounts = trips.agg(
    count_if(fare_amount < 0,                         "fare_negative"),
    count_if(total_amount < 0,                        "total_negative"),
    count_if((fare_amount < 0) | (total_amount < 0),  "either_negative"),
)

display(negative_amounts)
add_to_report("12. Negative amounts", negative_amounts)

#### 13. Negative rows that mirror a positive row (voids)
Same times, zones and vendor, opposite total amount → a reversal record, not a trip.

In [0]:
negative_rows = trips.filter(total_amount < 0).alias("neg")
positive_rows = trips.filter(total_amount > 0).alias("pos")

same_trip = [
    F.col("neg.tpep_pickup_datetime")  == F.col("pos.tpep_pickup_datetime"),
    F.col("neg.tpep_dropoff_datetime") == F.col("pos.tpep_dropoff_datetime"),
    F.col("neg.PULocationID").eqNullSafe(F.col("pos.PULocationID")),
    F.col("neg.DOLocationID").eqNullSafe(F.col("pos.DOLocationID")),
    F.col("neg.VendorID").eqNullSafe(F.col("pos.VendorID")),
    F.col("neg.total_amount") == -F.col("pos.total_amount"),
]

void_pairs = negative_rows.join(positive_rows, same_trip).agg(F.count("*").alias("mirrored_rows"))

display(void_pairs)
add_to_report("13. Negative rows that mirror a positive row", void_pairs)

### F. Categorical columns

#### 14. VendorID distribution

In [0]:
vendors = (
    trips
    .groupBy("VendorID")
    .agg(F.count("*").alias("trips"))
    .orderBy("VendorID")
)

display(vendors)
add_to_report("14. VendorID distribution", vendors)

#### 15. payment_type distribution

In [0]:
payment_types = (
    trips
    .groupBy("payment_type")
    .agg(F.count("*").alias("trips"))
    .orderBy("payment_type")
)

display(payment_types)
add_to_report("15. payment_type distribution", payment_types)

#### 16. NULL passenger_count, by vendor and payment type

In [0]:
null_passenger_count = (
    trips
    .filter(F.col("passenger_count").isNull())
    .groupBy("VendorID", "payment_type")
    .agg(F.count("*").alias("trips"))
    .orderBy(F.desc("trips"))
)

display(null_passenger_count)
add_to_report("16. NULL passenger_count", null_passenger_count)

### G. Distance

#### 17. trip_distance

In [0]:
distance_checks = trips.agg(
    count_if(trip_distance == 0,    "zero"),
    count_if(trip_distance < 0,     "negative"),
    count_if(trip_distance > 100,   "over_100_miles"),
    F.max(trip_distance).alias("max_miles"),
)

display(distance_checks)
add_to_report("17. trip_distance", distance_checks)

### Save the report

In [0]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
report_path = f"{OUTPUT_DIR}/profile_report.md"

with open(report_path, "w") as f:
    f.write("# Raw data profile — 01_bronze.yellow_trips_raw\n\n")
    f.write("\n".join(report.values()))

trips.unpersist()
print(f"Saved {report_path} ({len(report)} checks)")